In [1]:
# ============================================================
# TRANSFORM CONSTRUCTORS DATA — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [2]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/03_silver_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:45:52 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:45:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:45:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:45:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:45:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/09 13:45:53 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/09 13:45:53 

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [3]:
# -----------------------------------------
# 2. Receive p_batch_id (unified logic)
# -----------------------------------------

# Case 1: Papermill parameter
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# Case 2: Databricks widget
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# Case 3: Manual Jupyter execution → auto-detect latest batch
else:
    bronze_folder = f"{BRONZE_PATH}/constructors/data.parquet"
    batch_folders = [
        f.split("=")[1]
        for f in os.listdir(bronze_folder)
        if f.startswith("batch_id=")
    ]
    if not batch_folders:
        raise Exception("❌ No batch_id partitions found in Bronze constructors")
    p_batch_id = sorted(batch_folders)[-1]
    print("⚠️ Auto-selected latest batch_id:", p_batch_id)

# Final validation
if p_batch_id is None or p_batch_id == "":
    raise Exception("❌ p_batch_id not provided to Silver notebook")

print("Using p_batch_id:", p_batch_id)

⚠️ Auto-selected latest batch_id: 20260609_131752
Using p_batch_id: 20260609_131752


In [4]:
# -----------------------------------------
# 3. Define Bronze + Silver paths
# -----------------------------------------
bronze_path = f"{BRONZE_PATH}/constructors/data.parquet"
silver_path = f"{SILVER_PATH}/constructors"
os.makedirs(silver_path, exist_ok=True)

In [5]:
# -----------------------------------------
# 4. Read Bronze (ONLY this batch)
# -----------------------------------------
constructors_df = (
    spark.read.parquet(bronze_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Bronze constructors read")
constructors_df.show(5)

✔ Bronze constructors read
+-------------+----------+-----------+--------------------+--------------------+--------------------+---------------+
|constructorId|      name|nationality|                 url|    ingest_timestamp|         source_file|       batch_id|
+-------------+----------+-----------+--------------------+--------------------+--------------------+---------------+
|        adams|     Adams|   american|http://en.wikiped...|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|
|          afm|       AFM|     german|http://en.wikiped...|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|
|          ags|       AGS|     french|http://en.wikiped...|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|
|         alfa|Alfa Romeo|      swiss|http://en.wikiped...|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|
|   alphatauri|AlphaTauri|    italian|http://en.wikiped...|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|
+-------------+----------+---

In [6]:
# -----------------------------------------
# 5. Drop URL column
# -----------------------------------------
constructors_dropped_df = constructors_df.drop("url")

In [7]:
# -----------------------------------------
# 6. Standardize + rename columns
# -----------------------------------------
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("name", "constructor_name")
)

In [8]:
# -----------------------------------------
# 7. Remove duplicates
# -----------------------------------------
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [9]:
# -----------------------------------------
# 8. Title Case transformation
# -----------------------------------------
constructors_final_df = (
    constructors_distinct_df
        .withColumn("nationality", F.initcap("nationality"))
)

In [10]:
# -----------------------------------------
# 9. Write to Silver
# -----------------------------------------
write_to_silver(
    constructors_final_df,
    f"{silver_path}/data.parquet",
    merge_keys=["constructor_id"]
)

print("✔ Constructors Silver written:", f"{silver_path}/data.parquet")

✔ Constructors Silver written: /Users/manoharazalki/F1-Analytics/data/silver/constructors/data.parquet


In [11]:
# -----------------------------------------
# 10. Validate
# -----------------------------------------
spark.read.parquet(f"{silver_path}/data.parquet").show(5)

+--------------+----------------+-----------+--------------------+--------------------+---------------+--------------------+--------------------+
|constructor_id|constructor_name|nationality|    ingest_timestamp|         source_file|       batch_id|   created_timestamp|   updated_timestamp|
+--------------+----------------+-----------+--------------------+--------------------+---------------+--------------------+--------------------+
|         adams|           Adams|   American|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|2026-06-09 13:45:...|2026-06-09 13:45:...|
|           afm|             AFM|     German|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|2026-06-09 13:45:...|2026-06-09 13:45:...|
|           ags|             AGS|     French|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|2026-06-09 13:45:...|2026-06-09 13:45:...|
|          alfa|      Alfa Romeo|      Swiss|2026-06-09 13:17:...|/Users/manoharaza...|20260609_131752|2026-06-09 13:45:...|